<h1 align="center"><strong><font size="6"> Model Estimation SFMMO Dev </h1></strong></font>

<br>
<br>

**SFMMO Dev**: development code for the **Soccer Factor Model for Match Outcomes** -- based on **SFM I**.


**SFM I** is the model in [Andorra & Göbel (2024)](https://arxiv.org/abs/2412.05911) -- see notebook `013_SFM_Sloan__Submission.ipynb`.



- Target: Goals of EITHER Team -- i.e. 2 observations per match, and *home pitch* factor becomes explicit:
> - SFMMO DevA: point-diff; goals_scored*; goals_conceded*; MOM; FD_MOM; ELO
> - SFMMO DevB: tm_team; tm_opponent
> - SFMMO DevC: DevA + DevB
> - SFMMO DevD: DevA / {MOM__M; FD_MOM__M}
> - SFMMO DevE: DevA / {goals_scored*, goals_conceded*}
> - SFMMO DevF: DevA / {MOM, FD_MOM}
> - SFMMO DevG: DevA / {ELO}
> - SFMMO DevH: DevA / {FD_MOM}
> - SFMMO DevI: point-diff; ELO

<br>



In [1]:
# --- Connect to Google-Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# NOTE: order matters on Colab. jax[cuda] pulls its own dependency set, so pymc/numpyro/
# pytensor are installed LAST and pytensor is pinned -- otherwise the jax install can leave a
# pytensor version that breaks the numpyro NUTS backend. Do not re-shuffle these lines.
!pip install -qq plotly
#!pip install -qq preliz
!pip install -qq graphviz
!pip install -qq arviz arviz-plots
!pip install -qq jax[cuda] -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
!pip install -qq "pymc<6" numpyro pymc-bart "pytensor>=2.38.2,<2.39"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.8/394.8 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.9/179.9 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 514.3/514.3 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 127.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.8/237.8 kB 18.6 MB/s eta 0

In [3]:
from typing import List, Union


# --- Usual Libraries:
import numpy as np
import pandas as pd
from tqdm import tqdm

# --- PyMC & Affiliates
#import arviz as az
#import preliz as pz   # unused (pz.* never called) -- install commented out above
import pymc as pm
import pytensor.tensor as pt
import arviz as az
import arviz_plots as azp
import arviz_stats
import xarray as xr
import copy


# --- BART
import pymc_bart as pmb


# --- Stats
from scipy.stats import norm, nbinom, poisson
from sklearn.preprocessing import StandardScaler

# --- Interactive Plots:
import plotly.graph_objects as go
import plotly

# --- Plotting:
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import seaborn as sns

seed = sum(map(ord, "sfm"))
rng = np.random.default_rng(seed)

pm.__version__

'5.28.5'

In [4]:
np.__version__

'2.0.2'

In [5]:
# -------------------------------------- USER INTERACTION -------------------------------------- #

# --- Set the directory to the datafile ('SFM_data_byPlayer.csv'):
directory = '/content/drive/MyDrive/Colab Notebooks/51_SoccerAnalytics/'

# --- Which is the last season you want to Include in Training? [YYYY/YY]

dict_EW = {'trainT':['2019/20','2020/21','2021/22','2022/23'],
           'valT':['2020/21','2021/22','2022/23','2023/24']}

# --- HOLDOUT (M7.3): season(s) NO experiment may ever tune against. Scored EXACTLY ONCE,
#     at the end, for the single chosen configuration -- that number is the honest estimate
#     of next-season performance. 2024/25 is the natural choice: it is the most recent season
#     and is already untouched by every expanding-window fold above.
#     ALSO off-limits for tuning: the frozen WC2026 vintages in 10_data/106_Website/_vintages/
#     (they are a completed out-of-sample experiment; tuning on them destroys that status).
#     The asserts are the enforcement -- the receipts ethos applied to development.
HOLDOUT_SEASONS = ['2024/25']

# --- THE ONE HOLDOUT RUN. Setting RUN_HOLDOUT = True re-points the window at 2024/25 and
#     disarms the tripwire EXPLICITLY -- this is a conscious, once-only act. The selection is
#     already DONE (E chosen on the four folds, 2026-08-03, pre-registered rule); this number
#     is the honest performance estimate of the committed spec and CANNOT overturn the choice.
RUN_HOLDOUT = False

if RUN_HOLDOUT:
    dict_EW = {'trainT': ['2023/24'], 'valT': ['2024/25']}
    print('!' * 78)
    print('!!  HOLDOUT RUN -- burning 2024/25. This is done ONCE, for the committed spec.')
    print('!!  Selection is closed; whatever this number is, it is reported, not acted on.')
    print('!' * 78)
else:
    assert not (set(HOLDOUT_SEASONS) & set(dict_EW['valT'])),  'HOLDOUT leaked into the validation folds!'
    assert not (set(HOLDOUT_SEASONS) & set(dict_EW['trainT'])),'HOLDOUT leaked into the training-end list!'
    assert max(dict_EW['valT']) < min(HOLDOUT_SEASONS),        'a validation season sits at/after the HOLDOUT!'

# --- Cross-Sectional Standardization, or whole-sample Standardization?
do__scaleCS = True

# --- E3-B: Dixon-Coles low-score dependence (two-stage). Per fold: rho is fitted by ML on the
#     TRAINING matches (given the model's fitted lambdas), then applied to that fold's OOS joint
#     grids. Both variants are stored (Yhat = DC, Yhat_indep = independent) for the A/B table.
USE_DIXON_COLES = True

# --- Which Dev-Version?
#     'E' is the COMMITTED 2026/27 spec, chosen by the pre-registered variant re-trial on the
#     repaired harness (2026-08-03, 8 variants, 6,844 paired matches): momentum -- now that the
#     EWMA bug is fixed -- earns decisive readmission (A/C/E beat K at |t| >= 3), cum-goals are
#     fully redundant given momentum (A vs E: t = -0.2), and TM adds nothing (C~A, L~K).
#     E = points_diff + momentum (3 horizons, levels + FDs + interactions) + ELO.
#     History: 'K' (no momentum) was the WC-era choice -- selected while the momentum block was
#     broken and gamedays 1-2 were silently deleted; dethroned on fair re-trial.
devVersion = 'E'

# -------------------------------------- USER INTERACTION -------------------------------------- #

<br>

## 00 &emsp; Auxiliaries

In [6]:
def compute_log_likelihood(predictions, actuals):
    """
    predictions: array of shape (n_samples, n_categories)
                 e.g., [[0.4, 0.35, 0.2, 0.05], ...] for one match
    actuals: array of actual outcomes [0, 1, 2, 3, ...]
    """
    log_lik = 0
    for i, actual in enumerate(actuals):
        # Get probability assigned to actual outcome
        prob_actual = predictions[i][actual]
        log_lik += np.log(prob_actual + 1e-10)  # Add small epsilon to avoid log(0)

    return log_lik


def log_loss_categorical(probs, actuals, eps=1e-10):
    return -np.mean(np.log(probs[np.arange(len(actuals)), actuals] + eps))


def multi_class_brier_score(predictions, actuals, n_classes=4):
    """
    predictions: (n_samples, n_classes) probability matrix
    actuals: (n_samples,) actual outcomes
    """
    # Convert actuals to one-hot encoding
    one_hot = np.zeros((len(actuals), n_classes))
    one_hot[np.arange(len(actuals)), actuals] = 1

    # Compute Brier score
    brier = np.mean(np.sum((predictions - one_hot)**2, axis=1))
    return brier


def ranked_probability_score(predictions, actuals, n_classes=4):
    """
    Ranked Probability Score for ordinal outcomes
    """
    rps = 0
    for i, actual in enumerate(actuals):
        # Cumulative probabilities
        pred_cumsum = np.cumsum(predictions[i])

        # Actual cumulative (one-hot converted to cumulative)
        actual_cumsum = np.zeros(n_classes)
        actual_cumsum[actual:] = 1

        # RPS for this prediction
        rps += np.sum((pred_cumsum - actual_cumsum)**2)

    return rps / len(actuals)



def expected_calibration_error(predictions, actuals, n_bins=10):
    """
    ECE for probabilistic predictions
    """
    # Get predicted probabilities for actual class
    pred_probs = predictions[np.arange(len(actuals)), actuals]

    # Create bins
    bin_edges = np.linspace(0, 1, n_bins + 1)

    ece = 0
    for i in range(n_bins):
        # Find predictions in this bin
        in_bin = (pred_probs >= bin_edges[i]) & (pred_probs < bin_edges[i+1])

        if np.sum(in_bin) > 0:
            # Average predicted probability in bin
            avg_pred = np.mean(pred_probs[in_bin])

            # Actual accuracy in bin (for max predicted class)
            max_pred = np.argmax(predictions[in_bin], axis=1)
            avg_actual = np.mean(max_pred == actuals[in_bin])

            # Weighted by bin size
            ece += np.abs(avg_pred - avg_actual) * np.sum(in_bin)

    return ece / len(actuals)


def ordinal_accuracy(predictions, actuals):
    """
    Percentage of times the highest probability category was correct
    """
    pred_classes = np.argmax(predictions, axis=1)
    return np.mean(pred_classes == actuals)


def ordinal_mae(predictions, actuals):
    """
    MAE treating categories as ordinal
    """
    pred_classes = np.argmax(predictions, axis=1)
    return np.mean(np.abs(pred_classes - actuals))


def update_elo(r_home, r_away, result, K=20, home_adv=50):
    exp_home = 1 / (1 + 10 ** ((r_away - r_home - home_adv) / 400))
    if result == 2:    # home win
        s_home = 1.0
    elif result == 1:  # draw
        s_home = 0.5
    elif result == 0:  # away win
        s_home = 0.0
    else:
        raise ValueError("Invalid result value. Must be 0, 1, or 2.")

    r_home_new = r_home + K * (s_home - exp_home)
    r_away_new = r_away + K * ((1 - s_home) - (1 - exp_home))
    return r_home_new, r_away_new



def softmax(x):
    return np.exp(x)/sum(np.exp(x))

In [7]:
def rootogram(obs_counts, pp_counts, max_val=8, title=""):
    """
    obs_counts : 1D int array of observed goal counts        (n_fixtures,)
    pp_counts  : 2D int array of posterior predictive counts (n_fixtures, n_samples)
    """
    obs_counts = np.asarray(obs_counts, dtype=int)
    pp_counts  = np.asarray(pp_counts,  dtype=int)

    xs = np.arange(max_val + 1)
    obs_freq = np.bincount(obs_counts, minlength=max_val + 1)[:max_val + 1]

    n_samp = min(pp_counts.shape[1], 200)
    exp_freq = np.array([
        np.bincount(pp_counts[:, s], minlength=max_val + 1)[:max_val + 1]
        for s in range(n_samp)
    ])  # shape (n_samp, max_val+1)

    exp_mean = exp_freq.mean(axis=0)
    exp_low  = np.quantile(exp_freq, 0.03, axis=0)
    exp_up   = np.quantile(exp_freq, 0.97, axis=0)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(xs, np.sqrt(obs_freq), alpha=0.5, label="observed")
    ax.plot(xs, np.sqrt(exp_mean), "ro-", label="expected (PP mean)")
    ax.fill_between(xs, np.sqrt(exp_low), np.sqrt(exp_up),
                    color="red", alpha=0.2, label="94% HDI")
    ax.set_xlabel("goals"); ax.set_ylabel("√frequency"); ax.set_title(title)
    ax.legend(); plt.show()

    print(f"\nRMSE: {np.round(np.sqrt(np.mean((obs_freq - exp_mean))),3)}")
    print(f"Diff: {np.round((obs_freq - exp_mean) / obs_freq,2)}\n")

In [8]:
def fit_dc_rho(lam_h, lam_a, x, y):
    """Two-stage Dixon-Coles (1997): fit the low-score dependence rho by ML, GIVEN fitted
    per-match lambdas. tau == 1 outside the {0,1}x{0,1} cells, so only matches ending
    0-0 / 0-1 / 1-0 / 1-1 contribute to the likelihood. Bounds keep every tau cell positive.
    Returns (rho_hat, n_contributing_matches, (lo, hi) bounds)."""
    from scipy.optimize import minimize_scalar
    lam_h = np.asarray(lam_h, dtype=float); lam_a = np.asarray(lam_a, dtype=float)
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    m = (x <= 1) & (y <= 1)
    lh, la, xx, yy = lam_h[m], lam_a[m], x[m], y[m]
    lo = -1.0 / max(lh.max(), la.max()) + 1e-6            # 1 + lam*rho > 0
    hi = min(1.0, 1.0 / (lh * la).max()) - 1e-6           # 1 - lam_h*lam_a*rho > 0, rho < 1
    def _nll(r):
        t = np.ones_like(lh)
        t = np.where((xx == 0) & (yy == 0), 1 - lh * la * r, t)
        t = np.where((xx == 0) & (yy == 1), 1 + lh * r,      t)
        t = np.where((xx == 1) & (yy == 0), 1 + la * r,      t)
        t = np.where((xx == 1) & (yy == 1), 1 - r,           t)
        return -np.log(np.clip(t, 1e-12, None)).sum()
    res = minimize_scalar(_nll, bounds=(lo, hi), method='bounded')
    return float(res.x), int(m.sum()), (lo, hi)


def apply_dc_tau(joint_s, lam_h_s, lam_a_s, rho):
    """Apply the Dixon-Coles tau correction to ONE fixture's per-sample joint grid.
    joint_s: (n_samples, K, K) with axis -2 = HOME goals, axis -1 = AWAY goals;
    lam_h_s / lam_a_s: (n_samples,). tau preserves total mass analytically, so no renorm."""
    g = joint_s.copy()
    g[:, 0, 0] *= np.clip(1.0 - lam_h_s * lam_a_s * rho, 1e-12, None)
    g[:, 0, 1] *= np.clip(1.0 + lam_h_s * rho,           1e-12, None)
    g[:, 1, 0] *= np.clip(1.0 + lam_a_s * rho,           1e-12, None)
    g[:, 1, 1] *= (1.0 - rho)
    return g


def get__perMatch_jointPMF(eta, idx_home, idx_away, scale=None, k_max=15, return_lams=False):
    """
    For every row in the training set, return the posterior-predictive PMF
    P(Y = k) for k = 0..k_max with credible intervals.

    Returns a long-format DataFrame: (obs_id, k, mean, lo, hi).
    """

    # --- 1. Stack chain × draw → samples
    lam_samples = np.exp(eta.stack(samples=("chain", "draw")).values)    # (n_obs, n_samples)

    # --- Extract Home-Away Match-Ups
    lam_h = lam_samples[idx_home, :]            # (n_fixtures, n_samples)
    lam_a = lam_samples[idx_away, :]


    # --- Probability of Number of Goals to Evaluate
    N_goals = np.arange(k_max + 1)

    # --- 2. PMFs per sample
    if scale is not None:
      scale_samples = scale.stack(samples=("chain", "draw")).values  # (n_samples,)
      # nbinom.pmf with broadcasting: (k_max+1, n_fixtures, n_samples)
      pmf_h = nbinom.pmf(
          N_goals[:, None, None],
          n=scale_samples[None, None, :],
          p=scale_samples[None, None, :] / (scale_samples[None, None, :] + lam_h[None, :, :]),
      )  # (k_max+1, n_fixtures, n_samples)
      pmf_a = nbinom.pmf(
          N_goals[:, None, None],
          n=scale_samples[None, None, :],
          p=scale_samples[None, None, :] / (scale_samples[None, None, :] + lam_a[None, :, :]),
      )
    else:
      # poisson.pmf with broadcasting: (k_max+1, n_fixtures, n_samples)
      pmf_h = poisson.pmf(N_goals[:, None, None],mu=lam_h[None, :, :])
      pmf_a = poisson.pmf(N_goals[:, None, None],mu=lam_a[None, :, :])

    # --- 3. Outer product per fixture per sample → joint
    # einsum: 'h f s, a f s -> f s h a'
    joint = np.einsum("hfs,afs->fsha", pmf_h, pmf_a)

    if return_lams:
        # per-fixture, per-sample lambdas -- needed by apply_dc_tau (E3-B)
        return joint, lam_h, lam_a
    return joint



def get__home_WDL(matchup_jointPMF,cred_region=0.9):

  """
  matchup_jointPMF: (n_samples, K, K) joint PMF for one fixture
  """
  # Per-sample W/D/L
  diag_idx = np.arange(matchup_jointPMF.shape[-1])

  p_draw = matchup_jointPMF[:, diag_idx, diag_idx].sum(axis=-1)   # (n_samples,)
  p_loss = np.triu(matchup_jointPMF, k=1).sum(axis=(-2, -1))      # away > home (home loss)
  p_win  = np.tril(matchup_jointPMF, k=-1).sum(axis=(-2, -1))     # home > away

  alpha_lo = (1 - cred_region) / 2
  alpha_hi = 1 - alpha_lo

  df = pd.DataFrame(index=['W', 'D', 'L'], columns=['low', 'mid', 'up'], dtype=float)
  for label, arr in [('W', p_win), ('D', p_draw), ('L', p_loss)]:
      df.loc[label, 'mid'] = arr.mean()
      df.loc[label, 'low'] = np.quantile(arr, alpha_lo)
      df.loc[label, 'up']  = np.quantile(arr, alpha_hi)


  return df

## 1 &emsp; The Grand Loop

In [9]:
# ======================================== The Grand Loop ======================================== #

dict_preds = {key: {} for key in dict_EW['valT']}
dict_fitEval = {key: {} for key in dict_EW['valT']}



for t in range(len(dict_EW['trainT'])):



  # --- Extract Training-End and Validation Season:
  train_end = dict_EW['trainT'][t]
  val_seasons = [dict_EW['valT'][t]]

  # --------------------------------- 00. Data Preparation --------------------------------- #
  data_raw = pd.read_csv(f"{directory}/10_data/102_Development/data_byPlayer__SFM_II__TM.csv")
  #data_raw = pd.read_csv(f"{directory}/10_data/106_Website/data_byPlayer__SFM_II.csv")

  # --- Kick 'ligue-1': too few seasons in the training-set
  #data_raw = data_raw[~(data_raw['name_league'] == 'ligue-1')]

  # --- Pre-Processing, Part I:
  data_raw['kick_off'] = pd.to_datetime(data_raw['kick_off'])
  data_raw = data_raw.sort_values(['name_player','season','kick_off'])

  # --- Columns to Keep:
  keepCols = ['points_team','points_opp','goalsscored_inGame_team','goalsscored_inGame_opp',
            'goalsscored_cum_team','goalsscored_cum_opp','goalsconceded_cum_team','goalsconceded_cum_opp',
            'home_pitch','goalsscored_rank_team','goalsconceded_rank_opp',
            'id_match','name_team','name_opp','name_league','id_league','season','gameday','kick_off',
            'goalsscored_rank_opp','goalsconceded_rank_team','goalsscored_diff',
            'goal_balance_team','goal_balance_opp','goal_balance_diff','points_diff',
            'tm_marketvalue_team_squad','tm_marketvalue_opp_squad']

  # --- Extract unique Matches:
  complete_data = data_raw.drop_duplicates(subset=['id_match','home_pitch'])[keepCols].copy().sort_values(['name_league','kick_off']).reset_index(drop=True)

  # --- Keep only Matches for which we observe both Home & Away Team:
  complete_data = complete_data.loc[complete_data['id_match'].duplicated(keep=False),:]


  # Note:       We need two observations per match. This is preserved here.
  #             The home team is identified by the 'home_pitch' indicator.
  # Important:  Each observation is to be seen from the perspective of the TEAM --- but that does not say if TEAM plays at home!
  #             Hence, we can model the target easily, by taking the TEAM variable as the anchor!

  # --- Target: Number of Goals
  complete_data['match_outcome'] = complete_data['goalsscored_inGame_team'].copy()




  # -------------------------------------------- Feature Engineering -- Part I -------------------------------------------- #

  # --- Some Game-Day-Adjustments are necessary -- especially for cross-sectional scaling if a Gameday is '14.1', give it some grace and set it to '14'.
  complete_data['gameday_orig'] = complete_data['gameday'].copy()
  complete_data['gameday'] = [int(float(i.split('_')[1][2:])) for i in complete_data['id_match'].values]


  # -------------------------------------------- ELO Ratings -------------------------------------------- #
  complete_data[['elo_team','elo_opp']] = np.nan

  # --- For Compatibility:
  complete_data['match_outcome__home'] = 1
  complete_data.loc[complete_data['goalsscored_inGame_team'] > complete_data['goalsscored_inGame_opp'],'match_outcome__home'] = 2
  complete_data.loc[complete_data['goalsscored_inGame_team'] < complete_data['goalsscored_inGame_opp'],'match_outcome__home'] = 0




  # --- Run over Leagues
  N_leagues = complete_data['name_league'].unique().tolist()


  for ll in N_leagues:

    #print(f'\nELO-Ratings for league: {ll}')

    # --- Store the ELO ratings:
    ELO_rating = {key: 1500 for key in complete_data.loc[complete_data['name_league'] == ll,'name_team'].unique()}


    # --- Available Seasons:
    N_seasons = complete_data.loc[complete_data['name_league'] == ll,'season'].unique().tolist()


    # --- Run across the season:
    for ss in N_seasons:

      # --- Extract the season:
      ll_ss_data = complete_data[(complete_data['name_league'] == ll) & (complete_data['season'] == ss)].copy().sort_values('kick_off')


      # --- At the beginning of the season, adjust the rating:
      ss_idx = N_seasons.index(ss)
      if ss_idx > 0:

        # --- Teams in previous season:
        ss_t1__teams = complete_data.loc[(complete_data['name_league'] == ll) & (complete_data['season'] == N_seasons[ss_idx-1]),'name_team'].unique().tolist()

        for team in ELO_rating.keys():
          if team in ss_t1__teams:
            ELO_rating[team] = ELO_rating[team] * 0.75 + 1500 * 0.25
          else:
            ELO_rating[team] = 1300


      # --- Run across the season:
      for gg in range(ll_ss_data.shape[0]):

        # --- Home Team:
        gg_home = ll_ss_data['name_team'].iloc[gg]

        # --- Away Team:
        gg_away = ll_ss_data['name_opp'].iloc[gg]

        # --- Get Current ELO Ratings:
        gg_home__elo = ELO_rating[gg_home]
        gg_away__elo = ELO_rating[gg_away]

        # --- Insert Current ELO Ratings:
        gg_idx = ll_ss_data.index[gg]
        complete_data.loc[gg_idx,'elo_team'] = gg_home__elo
        complete_data.loc[gg_idx,'elo_opp'] = gg_away__elo

        # --- Update ELO Ratings:
        ELO_rating[gg_home], ELO_rating[gg_away] = update_elo(gg_home__elo,gg_away__elo,ll_ss_data['match_outcome__home'].iloc[gg])




  complete_data['elo_diff'] = complete_data['elo_team'] - complete_data['elo_opp']
  complete_data['elo_team_opp'] = complete_data['elo_team'] * complete_data['elo_opp']


  # ======================================== Feature Engineering --- Part II ======================================== #

  # --- Goal Appeal (of the match):
  complete_data['goal_appeal'] = complete_data['goalsconceded_rank_opp'] - complete_data['goalsscored_rank_team']



  # ---------------------- Team Momentum - exponentially-weighted MA of previous goals/outcomes ---------------------- #

  # --- For Compatibility: ---> BUT BE CAREFUL TO DROP IT AS IT IS JUST A HELPER!
  complete_data['home_team'] = complete_data['name_team'].copy()

  complete_data = complete_data.set_index(['kick_off','season','id_match','home_team'])
  complete_data[['teamMOM__S','teamMOM__M','teamMOM__L','oppMOM__S','oppMOM__M','oppMOM__L']] = np.nan
  complete_data[['FD_teamMOM__S','FD_teamMOM__M','FD_teamMOM__L','FD_oppMOM__S','FD_oppMOM__M','FD_oppMOM__L']] = np.nan

  for tt in complete_data['name_team'].unique():

    # --- Single-perspective rows ONLY: one row per match tt played, anchored on name_team.
    #     (The old (name_team==tt)|(name_opp==tt) filter pulled BOTH perspective rows of every
    #      match -- two rows per fixture -- so the EWMA ran over a doubled sequence: an effective
    #      halflife of half a match and aliased first-differences. Fix ported from SFMMOwm/006_050.)
    tt_data = complete_data.loc[complete_data['name_team'] == tt, :].copy()

    # --- Points from tt's own perspective (one row per match):
    tt_points = pd.DataFrame(index=tt_data.index)
    tt_points['points'] = tt_data['points_team']

    # --- FD of points
    tt_points['FD_points'] = tt_points.groupby(level='season')['points'].diff()
    # --- Calculate MOMENTUM: EWMA of points in previous games (halflife == one appearance)
    tt_points['MOM__S'] = tt_points.groupby(level='season')['FD_points'].ewm(halflife=1).mean().droplevel(level=0)
    tt_points['MOM__M'] = tt_points.groupby(level='season')['FD_points'].ewm(halflife=4).mean().droplevel(level=0)
    tt_points['MOM__L'] = tt_points.groupby(level='season')['FD_points'].ewm(halflife=8).mean().droplevel(level=0)

    # --- Calculate Delta MOMENTUM: first-difference of MOMENTUM
    tt_points['FD_MOM__S'] = tt_points.groupby(level='season')['MOM__S'].diff()
    tt_points['FD_MOM__M'] = tt_points.groupby(level='season')['MOM__M'].diff()
    tt_points['FD_MOM__L'] = tt_points.groupby(level='season')['MOM__L'].diff()

    # --- Team & Opponent Columns:
    team_cols = ['teamMOM__S','teamMOM__M','teamMOM__L','FD_teamMOM__S','FD_teamMOM__M','FD_teamMOM__L']
    opp_cols  = ['oppMOM__S','oppMOM__M','oppMOM__L','FD_oppMOM__S','FD_oppMOM__M','FD_oppMOM__L']
    src_cols  = ['MOM__S','MOM__M','MOM__L','FD_MOM__S','FD_MOM__M','FD_MOM__L']

    # --- Team: tt's momentum onto its own (name_team) rows
    complete_data.loc[tt_data.index, team_cols] = tt_points[src_cols].values

    # --- Opponent: map tt's momentum onto rows where tt is the opponent, via id_match
    #     (robust to row ordering -- replaces the fragile positional home==0 assignment)
    mom_by_match = tt_points[src_cols].copy()
    mom_by_match.index = tt_points.index.get_level_values('id_match')
    opp_rows = complete_data.index[complete_data['name_opp'] == tt]
    opp_ids  = opp_rows.get_level_values('id_match')
    complete_data.loc[opp_rows, opp_cols] = mom_by_match.loc[opp_ids].values


  # --- Some Two-Way Interactions:
  complete_data['teamMOM__S_L'] = complete_data['teamMOM__S'] * complete_data['teamMOM__L']
  complete_data['oppMOM__S_L'] = complete_data['oppMOM__S'] * complete_data['oppMOM__L']


  # --- Reset index & IMPORTANT: DROP 'home_team'
  complete_data = complete_data.reset_index().drop('home_team',axis=1)


  # ======================================== Feature Engineering -- Part III ======================================== #

  # -------------------------------------------- Transfermarket Market-Values -------------------------------------------- #

  # --- TM Ratio:
  complete_data['tm_marketvalue_ratio'] = complete_data['tm_marketvalue_team_squad'] / complete_data['tm_marketvalue_opp_squad']

  # --- Log-Transform the Raw Values
  complete_data['tm_marketvalue_team_squad'] = np.where(complete_data['tm_marketvalue_team_squad'] <= 0, 0, np.log(complete_data['tm_marketvalue_team_squad']))
  complete_data['tm_marketvalue_opp_squad'] = np.where(complete_data['tm_marketvalue_opp_squad'] <= 0, 0, np.log(complete_data['tm_marketvalue_opp_squad']))


  # ======================================== Define the Factors ======================================== #


  if devVersion in ['A']:

    # --- Numerical Factors:
    factors_CS = ['points_diff',
                  'goalsscored_cum_team','goalsscored_cum_opp',
                  'goalsconceded_cum_team','goalsconceded_cum_opp',
                  'teamMOM__S','oppMOM__S',
                  'teamMOM__M','oppMOM__M',
                  'teamMOM__L','oppMOM__L',
                  'teamMOM__S_L', 'oppMOM__S_L',
                  'FD_teamMOM__S','FD_oppMOM__S',
                  'FD_teamMOM__M','FD_oppMOM__M',
                  'FD_teamMOM__L','FD_oppMOM__L',
                  'elo_team','elo_opp',
                  'elo_team_opp'
                  ]

  elif devVersion in ['B']:

    factors_CS = ['tm_marketvalue_team_squad','tm_marketvalue_opp_squad'] #,'tm_marketvalue_ratio']

  elif devVersion in ['C']:

    factors_CS = ['points_diff',
                  'goalsscored_cum_team','goalsscored_cum_opp',
                  'goalsconceded_cum_team','goalsconceded_cum_opp',
                  'teamMOM__S','oppMOM__S',
                  'teamMOM__M','oppMOM__M',
                  'teamMOM__L','oppMOM__L',
                  'teamMOM__S_L', 'oppMOM__S_L',
                  'FD_teamMOM__S','FD_oppMOM__S',
                  'FD_teamMOM__M','FD_oppMOM__M',
                  'FD_teamMOM__L','FD_oppMOM__L',
                  'elo_team','elo_opp',
                  'elo_team_opp',
                  'tm_marketvalue_team_squad','tm_marketvalue_opp_squad']

  elif devVersion in ['D']:

    # --- Numerical Factors:
    factors_CS = ['points_diff',
                  'teamMOM__S','oppMOM__S',
                  'teamMOM__M','oppMOM__M',
                  'teamMOM__L','oppMOM__L',
                  'teamMOM__S_L', 'oppMOM__S_L',
                  'FD_teamMOM__S','FD_oppMOM__S',
                  'FD_teamMOM__M','FD_oppMOM__M',
                  'FD_teamMOM__L','FD_oppMOM__L',
                  'elo_team','elo_opp',
                  'elo_team_opp'
                  ]

  elif devVersion in ['E']:

    # --- Numerical Factors:
    factors_CS = ['points_diff',
                  'goalsscored_cum_team','goalsscored_cum_opp',
                  'goalsconceded_cum_team','goalsconceded_cum_opp',
                  'teamMOM__S','oppMOM__S',
                  #'teamMOM__M','oppMOM__M',
                  'teamMOM__L','oppMOM__L',
                  'teamMOM__S_L', 'oppMOM__S_L',
                  'FD_teamMOM__S','FD_oppMOM__S',
                  #'FD_teamMOM__M','FD_oppMOM__M',
                  'FD_teamMOM__L','FD_oppMOM__L',
                  'elo_team','elo_opp',
                  'elo_team_opp'
                  ]

  elif devVersion in ['F']:

    # --- Numerical Factors:
    factors_CS = ['points_diff',
                  'goalsscored_cum_team','goalsscored_cum_opp',
                  'goalsconceded_cum_team','goalsconceded_cum_opp',
                  #'teamMOM__S','oppMOM__S',
                  #'teamMOM__M','oppMOM__M',
                  #'teamMOM__L','oppMOM__L',
                  #'teamMOM__S_L', 'oppMOM__S_L',
                  #'FD_teamMOM__S','FD_oppMOM__S',
                  #'FD_teamMOM__M','FD_oppMOM__M',
                  #'FD_teamMOM__L','FD_oppMOM__L',
                  'elo_team','elo_opp',
                  'elo_team_opp'
                  ]

  elif devVersion in ['G']:

    # --- Numerical Factors:
    factors_CS = ['points_diff',
                  'goalsscored_cum_team','goalsscored_cum_opp',
                  'goalsconceded_cum_team','goalsconceded_cum_opp',
                  'teamMOM__S','oppMOM__S',
                  'teamMOM__M','oppMOM__M',
                  'teamMOM__L','oppMOM__L',
                  'teamMOM__S_L', 'oppMOM__S_L',
                  'FD_teamMOM__S','FD_oppMOM__S',
                  'FD_teamMOM__M','FD_oppMOM__M',
                  'FD_teamMOM__L','FD_oppMOM__L'
                  #'elo_team','elo_opp',
                  #'elo_team_opp'
                  ]

  elif devVersion in ['H']:

    # --- Numerical Factors:
    factors_CS = ['points_diff',
                  'goalsscored_cum_team','goalsscored_cum_opp',
                  'goalsconceded_cum_team','goalsconceded_cum_opp',
                  'teamMOM__S','oppMOM__S',
                  'teamMOM__M','oppMOM__M',
                  'teamMOM__L','oppMOM__L',
                  'teamMOM__S_L', 'oppMOM__S_L',
                  #'FD_teamMOM__S','FD_oppMOM__S',
                  #'FD_teamMOM__M','FD_oppMOM__M',
                  #'FD_teamMOM__L','FD_oppMOM__L'
                  'elo_team','elo_opp',
                  'elo_team_opp'
                  ]

  elif devVersion in ['I']:

    # --- Numerical Factors:
    factors_CS = ['points_diff',
                  'elo_team','elo_opp',
                  'elo_team_opp'
                  ]

  elif devVersion in ['K']:

    # --- PRODUCTION set (M5). Identical to the deployed SFMMOwm Model K in
    #     006_050__Predictions_MatchOutcome__SFMMOwm.py: points_diff + cumulative goals
    #     + ELO LEVELS. No momentum (inert across A-H, and the block was buggy until now),
    #     no elo_diff / elo_team_opp (redundant given the two ELO levels, and the WC knockout
    #     showed correlated form features reorder the board via large offsetting betas).
    #     Bonus: no first-difference features => no undefined cells => gamedays 1-2 survive (M1).
    factors_CS = ['points_diff',
                  'goalsscored_cum_team','goalsscored_cum_opp',
                  'goalsconceded_cum_team','goalsconceded_cum_opp',
                  'elo_team','elo_opp'
                  ]

  elif devVersion in ['L']:

    # --- L = K + Transfermarkt squad values (log). Motivated by the E3-B finding: the market's
    #     remaining edge is INFORMATION, not statistics -- TM values are the one feature carrying
    #     outside information (squad quality as priced by the transfer market). All other
    #     challengers recombine information K already has.
    factors_CS = ['points_diff',
                  'goalsscored_cum_team','goalsscored_cum_opp',
                  'goalsconceded_cum_team','goalsconceded_cum_opp',
                  'elo_team','elo_opp',
                  'tm_marketvalue_team_squad','tm_marketvalue_opp_squad'
                  ]



  # --- Other Factors:
  other_factors = ['home_pitch']


  # --- Concatenate:
  factors = other_factors + factors_CS

  # ======================================== Target & ID vars ======================================== #

  IDvar = ['id_match','name_team','name_opp','name_league','id_league','season','gameday','kick_off']
  Yvar = 'match_outcome'




  # ======================================== Some Data Preprocessing ======================================== #


  # --- 0.0 Goal-count cap (M2). RESOLUTION: NO CAP AT ALL.
  #     The original code capped at 5 under an UNCENSORED Poisson, which biases lambda DOWN for
  #     exactly the strong attacks (~14% of a lam=3.5 side's mass sits above 5) -- a mechanical
  #     contributor to the "under-confidence on favourites" (L2). The honest way to KEEP a cap is
  #     a right-censored likelihood, but pm.Censored needs the Poisson log-CDF, whose gradient is
  #     NaN under the numpyro/JAX backend this notebook samples with -> 100% divergences.
  #     (It samples fine on PyMC's default C backend, which is why the local smoke test missed it:
  #     ALWAYS smoke-test on the sampler the notebook actually uses.)
  #     Measured on the real data (72,726 team-match rows, max = 10 goals): only 414 rows (0.57%)
  #     exceed 5 and 32 rows (0.04%) exceed 7. The cap buys almost no robustness, so the simplest
  #     correct answer is not to cap: no bias, no censoring, no JAX problem. k_max=15 covers the
  #     tail in the scoreline grid. Constant kept (documents the decision + allows re-enabling).
  GOALS_CAP = None
  complete_data['match_outcome__orig'] = complete_data['match_outcome'].copy()
  if GOALS_CAP is not None:
      complete_data['match_outcome'] = np.where(complete_data['match_outcome'] > GOALS_CAP, GOALS_CAP, complete_data['match_outcome'])

  # --- 0.0 Special Treatment:
  # --- M1: drop only on TARGET/IDs. The old blanket .dropna() deleted every row with any
  #     undefined factor -- which on real data was 100% of gamedays 1-2 (3,776 matches; the
  #     FD_*MOM first-differences are undefined for a team's first two appearances of a season).
  #     The cold-start regime was therefore absent from BOTH training and validation, which is
  #     precisely why the WC MD1 hole (log-loss worse than uniform) was invisible in development.
  #     Undefined form is now carried through and set to league-average AFTER scaling (see below).
  complete_data = complete_data[IDvar + [Yvar, 'match_outcome__orig'] + factors].dropna(subset=IDvar + [Yvar]).reset_index(drop=True)


  # --- 0.1 Validation Set:
  data_oos = complete_data.loc[complete_data['season'].isin(val_seasons),:].dropna(subset=IDvar + [Yvar]) # M1
  data_oos = data_oos.loc[data_oos['name_league'].isin(complete_data.loc[complete_data['season'] <= train_end,'name_league']),:]

  # --- 0.2 Get the Training-Data only:
  complete_data = complete_data.loc[complete_data['season'] <= train_end,:].dropna(subset=IDvar + [Yvar]) # M1, IDvar + [Yvar] + factors].dropna()

  # --- 0.3 Kick the first Season by League:
  complete_data = complete_data.loc[complete_data.groupby('name_league')['season'].transform('min') != complete_data['season'],:]


  # --- 0.4 Some Type-Setting:
  complete_data["kick_off"] = pd.to_datetime(
      complete_data["kick_off"], yearfirst=True
  ).dt.normalize()


  # --- 0.5 Final Sorting for Convenience:
  complete_data = complete_data.sort_values(["name_league", "kick_off"]).reset_index(
      drop=True
  )



  # ======================================== Cross-Sectional Standardization (by Gameday) ======================================== #

  if do__scaleCS:

    # --- For future use:
    train_means = complete_data.groupby('gameday')[factors_CS].mean()
    train_stds  = complete_data.groupby('gameday')[factors_CS].std()

    # --- Conduct actual scaling:
    data__scaleCS = complete_data.groupby('gameday')[factors_CS].apply(lambda x: (x - x.mean()) / x.std()).reset_index().set_index('level_1').drop('gameday',axis=1)

    #print('\nScaling data cross-sectionally!\n')


    # --- Merge:
    complete_data[factors_CS] = data__scaleCS


  # --- M1: explicit + AUDITED missing-factor policy (replaces the silent .dropna()).
  #     In STANDARDIZED space 0 == "league average", so an undefined feature contributes nothing
  #     -- the same thing the production pipeline does for a fixture with no form yet. Filling
  #     here (not before scaling) is essential: at gameday 1 the whole cross-section is undefined,
  #     so a pre-scaling fill would give std == 0 and reintroduce NaNs.
  # A zero cross-sectional std (e.g. gameday 1, where every team's cum-goals are identical)
  # yields +/-inf rather than NaN, which fillna would NOT catch -- normalize those first.
  complete_data[factors_CS] = complete_data[factors_CS].replace([np.inf, -np.inf], np.nan)
  _na = complete_data[factors_CS].isna()
  if _na.any().any():
      _audit = _na.groupby(complete_data['gameday']).sum()
      _audit = _audit.loc[_audit.sum(axis=1) > 0]
      print(f"[M1 audit | train] {int(_na.sum().sum())} undefined factor-cells across "
            f"{int(_na.any(axis=1).sum())} rows -> set to league average. By gameday (head):")
      print(_audit.loc[:, _audit.sum() > 0].head(5).to_string())
  complete_data[factors_CS] = complete_data[factors_CS].fillna(0.0)

  # --- Any row still carrying a NaN (target/ID/non-CS factor) is a genuine hole: drop loudly.
  _resid = complete_data.isna().any(axis=1).sum()
  if _resid:
      print(f"[M1 audit | train] dropping {int(_resid)} rows with non-factor NaNs (genuine holes)")
  complete_data = complete_data.dropna()


  # ======================================== 1.1 Factor Standardization ======================================== #

  if not do__scaleCS:
      factors_CS_train = complete_data[factors_CS].copy()

      # --- Do the Standardization
      scaler = StandardScaler()
      factors_CS_sdz = pd.DataFrame(
          scaler.fit_transform(factors_CS_train), columns=factors_CS
      )

      # --- Add the non-numeric factor to the standardized DataFrame
      factors_sdz = factors_CS_sdz.copy()
      factors_sdz[other_factors] = complete_data[other_factors].copy()

      # --- Ensure that the order is the same as the PyMC coords later on
      factors_sdz = factors_sdz[factors]


      #print('\nScaling data across the Whole Sample!\n')


  else:

      # --- Standardization already done! Ensure that the order is the same as the PyMC coords later on:
      factors_sdz = complete_data[factors].copy()

      #print('\nData already scaled cross-sectionally!\n')



  # =========================== Changing Variables for Train- & Test-Set =========================== #

  # --- Teams:
  names_teams = sorted(set(complete_data["name_team"]).union(complete_data["name_opp"]))
  team_to_idx = {t: i for i, t in enumerate(names_teams)}


  # --- Home & Away Team Indices:
  idx_home = complete_data[complete_data['home_pitch'] == 1].index
  idx_away = complete_data[complete_data['home_pitch'] == 0].index


  # --- Global Factors
  factors_g = [f for f in factors if f != 'home_pitch']

  # --- Set the Coords:
  COORDS = {
      "factor_g": factors_g,
      "obs_id": complete_data.index,
      "teams": names_teams
  }


  # ========================================= The Team-Factors ========================================= #


  with pm.Model(coords=COORDS) as SFMMO__dev:

      # --- Set the Data:
      X_gf = pm.Data(
          "X_gf", factors_sdz[factors_g].copy().to_numpy(), dims=("obs_id", "factor_g")
      )
      X_home = pm.Data(
          "X_home", factors_sdz['home_pitch'].copy().to_numpy().astype(int), dims="obs_id"
      )
      Y = pm.Data(
          "Y", complete_data[Yvar].copy().to_numpy(), dims="obs_id"
      )
      idx_team = pm.Data("idx_team", complete_data["name_team"].map(team_to_idx).to_numpy(), dims="obs_id")
      idx_opp = pm.Data("idx_opp", complete_data["name_opp"].map(team_to_idx).to_numpy(), dims="obs_id")
      idx_home = pm.Data("idx_home", (complete_data["home_pitch"] == 1).to_numpy(), dims="obs_id")


      # --- Set the Model Priors:

      if devVersion in ['Z']:

        # --- BART:
        gX = pmb.BART('gX', X, Y ,m=50,shape=(2,'obs_id'))

      else:


        # --- Global Intercept: explicit, with a goals-scale prior (M3).
        #     Previously ABSENT -- the baseline log-rate was identified only through the drift
        #     of the free-Normal delta against its own shrinkage prior (implicit intercept);
        #     the SFMMOwm 7.7% lambda-deflation bug traced to exactly this ambiguity.
        mu = pm.Normal("mu", mu=np.log(complete_data[Yvar].mean()), sigma=0.2)

        # --- Team-Level Attack / Defense Fixed-Effects: BOTH zero-sum, FIXED scale 0.30.
        #     Ported from SFMMOwm FinalK (patch 3): symmetric identification (the old spec had
        #     alpha zero-sum but delta free-Normal), no hierarchical funnel, and the fixed scale
        #     is data-verified (posterior sigma_alpha~0.25, sigma_delta~0.31 under the old
        #     hierarchical fit). Allows target_accept 0.9 instead of 0.99.
        alpha = pm.ZeroSumNormal("alpha", sigma=0.30, dims="teams")
        delta = pm.ZeroSumNormal("delta", sigma=0.30, dims="teams")

        # --- Team-Level Home Advantage:
        mu_gamma    = pm.Normal("mu_gamma", mu=0.30, sigma=0.20)
        sigma_gamma = pm.HalfNormal("sigma_gamma", sigma=0.10)
        # --- --- Non-centered for sampling stability:
        gamma_raw   = pm.Normal("gamma_raw", 0, 1, dims="teams")
        beta_home  = pm.Deterministic(
            "beta_home",
            mu_gamma + gamma_raw * sigma_gamma,
            dims="teams",
        )

        # --- Factors are pooled across teams
        beta = pm.Normal("beta", mu=0, sigma=0.3, dims="factor_g")




  # ==================== Set the Model: Bayesian Time-Series Regression (Classificatuion) ==================== #


  with SFMMO__dev:


      # --- Conditional Mean:
      eta = pm.Deterministic('eta', mu + alpha[idx_team] - delta[idx_opp] + X_home * beta_home[idx_team] + pt.dot(X_gf, beta.T))

      if 1==2:
        # --- Variance:
        scale = pm.Gamma("scale", alpha=20, beta=0.1)


        # --- Likelihood:
        pm.NegativeBinomial("match_outcome", mu=pm.math.exp(eta), alpha=scale, observed=Y, dims="obs_id")

      else:

        # --- Likelihood:
        pm.Poisson("match_outcome", mu=pm.math.exp(eta), observed=Y, dims="obs_id")


  # ============================================= Inference! ============================================= #


  # --- In the paper we use 4 chains
  # --- Just for speed: set it to 1
  N_chains = 2

  with SFMMO__dev:

      if devVersion in ['Z']:

        idata = pm.sample(draws=4000, tune=1000, chains=N_chains,cores=4, random_seed=42, idata_kwargs={"log_likelihood": True})

      else:

        idata = pm.sample(nuts_sampler="numpyro",
                          random_seed=seed,   # M7: reported numbers must be reproducible
                          target_accept=0.9,    # 0.99 unnecessary after the fixed-scale ZSN reparam (SFMMOwm patch 3)
                          chains=N_chains,
                          draws=4000, tune=1000,
                          cores=1,  # --- CPU cores (irrelevant for GPU)
                          nuts_sampler_kwargs={"chain_method": "vectorized"},
                          idata_kwargs={"log_likelihood": True})  # --- Runs chains in parallel on GPU)



  # ============================= Some Preparation for Post-Estimation Processing ============================= #

  with SFMMO__dev:
      idata.extend(pm.sample_posterior_predictive(idata, random_seed=seed))




  # ===================================== OOS-Data: Preparation ===================================== #


  # --- 0.1 Some Type-Setting:
  data_oos["kick_off"] = pd.to_datetime(
      data_oos["kick_off"], yearfirst=True
  ).dt.normalize()


  # --- 0.2 Final Sorting for Convenience:
  data_oos = data_oos.sort_values(["name_league", "kick_off"]).reset_index(
      drop=True
  )

  if do__scaleCS:
    #data__scaleCS = data_oos.groupby('gameday')[factors_CS].apply(lambda x: (x - x.mean()) / x.std()).reset_index().set_index('level_1').drop('gameday',axis=1)

    # --- Merge:
    #data_oos[factors_CS] = data__scaleCS
    # M1: a gameday whose training cross-section has ZERO variance in a factor (e.g. gameday 1,
    # where every team's cum-goals/points-diff are identical) carries no information. The row-wise
    # apply below builds an OBJECT-dtype row, so 0/0 RAISES ZeroDivisionError rather than giving
    # NaN -- hence sanitize the divisor first; the fill then maps these to league average.
    _train_stds_safe = train_stds.replace(0.0, np.nan)
    data_oos[factors_CS] = data_oos.apply(lambda row: (row[factors_CS] - train_means.loc[row['gameday']]) / _train_stds_safe.loc[row['gameday']], axis=1)

    # --- M1: same missing-factor policy as training (parity). Undefined form -> league average.
    data_oos[factors_CS] = data_oos[factors_CS].replace([np.inf, -np.inf], np.nan)   # std==0 guard
    _na_oos = data_oos[factors_CS].isna()
    if _na_oos.any().any():
        print(f"[M1 audit | oos] {int(_na_oos.sum().sum())} undefined factor-cells across "
              f"{int(_na_oos.any(axis=1).sum())} rows -> set to league average.")
    data_oos[factors_CS] = data_oos[factors_CS].fillna(0.0)


    print('\nScaling Data Cross-Sectionally!\n')


  else:

      # --- Standardization already done! Ensure that the order is the same as the PyMC coords later on:
      print('\nNo Sclaing Applied!\n')


  factors_sdz__oos = data_oos[factors].copy()



  # ================================== Out-of-Sample Prediction ================================== #


  # --- Teams:
  names_teams__OOS = sorted(set(data_oos["name_team"]).union(data_oos["name_opp"]))
  team_to_idx__OOS = {t: i for i, t in enumerate(names_teams__OOS)}

  # --- Home & Away Team Indices:
  idx_home__OOS = data_oos[data_oos['home_pitch'] == 1].index
  idx_away__OOS = data_oos[data_oos['home_pitch'] == 0].index



  # ==================================== OOS: New & Existing Teams ==================================== #

  # --- Identify the Teams:
  teams_in_oos = set(data_oos["name_team"]).union(data_oos["name_opp"])
  truly_new_teams = sorted(t for t in teams_in_oos if t not in names_teams)

  # --- Combined coord: training teams first, then new teams
  names_teams_all = list(names_teams) + truly_new_teams
  team_to_idx_all = {t: i for i, t in enumerate(names_teams_all)}

  n_train = len(names_teams)        # original count
  n_new   = len(truly_new_teams)    # 0 if no new teams this OOS window


  # ---------------------------- Update the Model ---------------------------- #
  with SFMMO__dev:

      # New coords
      if n_new > 0:
          SFMMO__dev.add_coord("team__new", truly_new_teams)
      SFMMO__dev.add_coord("obs_id__oos", data_oos.index)

      # OOS data containers
      X_gf_oos    = pm.Data("X_gf_oos",
                            factors_sdz__oos.loc[data_oos.index, factors_g].to_numpy(),
                            dims=("obs_id__oos", "factor_g"))
      X_home_oos  = pm.Data("X_home_oos",
                            factors_sdz__oos.loc[data_oos.index, "home_pitch"].to_numpy().astype(int),
                            dims="obs_id__oos")
      idx_team_oos = pm.Data("idx_team_oos",
                            data_oos["name_team"].map(team_to_idx_all).to_numpy(),
                            dims="obs_id__oos")
      idx_opp_oos  = pm.Data("idx_opp_oos",
                            data_oos["name_opp"].map(team_to_idx_all).to_numpy(),
                            dims="obs_id__oos")
      idx_home_oos = pm.Data("idx_home_oos",
                            (data_oos["home_pitch"] == 1).to_numpy().astype(int),
                            dims="obs_id__oos")
      Y_oos        = pm.Data("Y_oos",
                            data_oos[Yvar].to_numpy().astype(int),
                            dims="obs_id__oos")

      # ============ New-team parameters (only if any) ============
      if n_new > 0:
          # Reference shared hyperparameters and globals
          sigma_alpha_hp = 0.30   # fixed ZSN scale (M3) -- the hyper-RVs no longer exist
          sigma_delta_hp = 0.30
          mu_gamma_hp    = SFMMO__dev.mu_gamma
          sigma_gamma_hp = SFMMO__dev.sigma_gamma

          # Empirical anchors (from earlier — or your heuristic if you prefer)
          # alpha_anchor, delta_anchor = ...
          alpha_anchor    = idata['posterior']['alpha'].stack(samples=('chain','draw')).median('samples').quantile(0.25).to_numpy()
          delta_anchor    = idata['posterior']['delta'].stack(samples=('chain','draw')).median('samples').quantile(0.25).to_numpy()

          alpha__new = pm.Normal("alpha__new",
                                mu=alpha_anchor, sigma=sigma_alpha_hp,
                                dims="team__new")
          delta__new = pm.Normal("delta__new",
                                mu=delta_anchor, sigma=sigma_delta_hp,
                                dims="team__new")
          gamma_raw__new = pm.Normal("gamma_raw__new", 0, 1, dims="team__new")
          beta_home__new = pm.Deterministic("beta_home__new",
                                            mu_gamma_hp + gamma_raw__new * sigma_gamma_hp,
                                            dims="team__new")

          # Combined parameter vectors via concatenation
          alpha_all      = pt.concatenate([alpha,      alpha__new])
          delta_all      = pt.concatenate([delta,      delta__new])
          beta_home_all  = pt.concatenate([beta_home,  beta_home__new])
      else:
          # No new teams this OOS window — combined is just the trained
          alpha_all      = alpha
          delta_all      = delta
          beta_home_all  = beta_home

      # ============ Single OOS linear predictor and likelihood ============
      eta_oos = pm.Deterministic(
          "eta_oos",
          mu
        + alpha_all[idx_team_oos]
        - delta_all[idx_opp_oos]
        + X_home_oos * beta_home_all[idx_team_oos]
        + pt.dot(X_gf_oos, beta),
          dims="obs_id__oos",
      )

      if 1==2:
        pm.NegativeBinomial(
            "match_outcome_oos",
            mu=pm.math.exp(eta_oos),
            alpha=scale,
            observed=Y_oos,
            dims="obs_id__oos",
        )
      else:
        pm.Poisson(
            "match_outcome_oos",
            mu=pm.math.exp(eta_oos),
            observed=Y_oos,
            dims="obs_id__oos",
        )

  # ============ One prediction call for everything ============
  with SFMMO__dev:
      oos_preds = pm.sample_posterior_predictive(
          idata,
          random_seed=seed,
          predictions=True,
          var_names=["eta_oos", "match_outcome_oos"]
                    + (["alpha__new", "delta__new", "beta_home__new"] if n_new > 0 else []),
          compile_kwargs={"mode": "NUMBA"},
      )



  # ===================================== Collect Results: Match-Up Goals Matrix ===================================== #


  # --- Assemble the Match-IDs and corresponding Match-Ups:
  Y__eval = data_oos[['id_match','name_team','name_opp','home_pitch','kick_off', 'match_outcome__orig'] + [Yvar]].copy().reset_index(names='index__data_oos')

  # --- Get Home & Away Indices within the Samples:
  idxSamples_home = Y__eval.loc[Y__eval['home_pitch'] == 1,'id_match'].reset_index().set_index('id_match')
  idxSamples_away = Y__eval.loc[Y__eval['home_pitch'] == 0,'id_match'].reset_index().set_index('id_match')

  idxSamples = pd.merge(idxSamples_home,idxSamples_away,left_index=True,right_index=True, suffixes=('__home','__away'))
  idxSamples_home = idxSamples['index__home'].values
  idxSamples_away = idxSamples['index__away'].values

  # ------------------------- E3-B: fit Dixon-Coles rho on TRAINING ------------------------- #
  #   Two-stage: rho by ML given the model's fitted lambdas, TRAINING matches of THIS fold only
  #   (never the validation season -- prospective discipline). lam uses exp(posterior-mean eta):
  #   the Jensen gap vs mean(exp(eta)) is immaterial for the rho likelihood and 8000x cheaper.
  rho_hat = None
  if USE_DIXON_COLES:
    _ih = complete_data.loc[complete_data['home_pitch'] == 1, 'id_match'].reset_index().set_index('id_match')
    _ia = complete_data.loc[complete_data['home_pitch'] == 0, 'id_match'].reset_index().set_index('id_match')
    _pair_tr = pd.merge(_ih, _ia, left_index=True, right_index=True, suffixes=('_home', '_away'))
    _lam_tr = np.exp(idata['posterior']['eta'].mean(dim=('chain', 'draw')).values)
    _xh = complete_data.loc[_pair_tr['index_home'].values, 'match_outcome'].values
    _ya = complete_data.loc[_pair_tr['index_away'].values, 'match_outcome'].values
    rho_hat, _n_ls, _bnds = fit_dc_rho(_lam_tr[_pair_tr['index_home'].values],
                                       _lam_tr[_pair_tr['index_away'].values], _xh, _ya)
    print(f"[E3-B] rho fitted on {len(_pair_tr):,} training matches "
          f"({_n_ls:,} in the low-score cells): rho_hat = {rho_hat:+.4f}  bounds {_bnds[0]:+.3f}..{_bnds[1]:+.3f}")

  # --- Get the Joint Posterior of the Goal-Matrix (+ per-sample lambdas for the tau correction)
  jointPMF, lamPMF_h, lamPMF_a = get__perMatch_jointPMF(eta=oos_preds['predictions']['eta_oos'],
                                    #scale=idata['posterior']['scale'],
                                    idx_home=idxSamples_home, idx_away=idxSamples_away,
                                    k_max=15,   # M2: 5 truncated up to ~11% of joint mass on lopsided fixtures
                                    return_lams=True)



  # ===================================== Evaluate Predictions ===================================== #

  # --- Accumulate in lists and build each frame ONCE. The previous version concatenated a
  #     one-row DataFrame per match: O(n^2), and it emitted a pandas FutureWarning (concat with
  #     empty/all-NA entries) that becomes an error in a future pandas. Behaviour is identical.
  _rows_Y, _rows_Yhat, _rows_Yhat_dc = [], [], []

  for m in tqdm(range(len(idxSamples_home))):

    # --- Observed Outcome:
    m_Y = Y__eval.loc[(idxSamples_home[m],idxSamples_away[m]),:].copy().sort_values('home_pitch',ascending=False)
    _d = m_Y['match_outcome__orig'].diff().values[1]
    m_Y__outcome = 2 if _d < 0 else 0 if _d > 0 else 1
    _rows_Y.append({'id_match': m_Y['id_match'].iloc[0], 'match_outcome': m_Y__outcome})

    # --- Predicted Probabilities: independent grid, and (E3-B) the tau-corrected grid
    _rows_Yhat.append(get__home_WDL(jointPMF[m,:,:,:])['mid'].iloc[::-1].values)
    if rho_hat is not None:
      _g = apply_dc_tau(jointPMF[m,:,:,:], lamPMF_h[m,:], lamPMF_a[m,:], rho_hat)
      _rows_Yhat_dc.append(get__home_WDL(_g)['mid'].iloc[::-1].values)

  Y__SFMMO = pd.DataFrame(_rows_Y, columns=['id_match','match_outcome'])
  Yhat_indep = pd.DataFrame(_rows_Yhat, columns=['0','1','2'])
  # 'Yhat' stays the headline object downstream: DC when enabled, else independent
  Yhat = pd.DataFrame(_rows_Yhat_dc, columns=['0','1','2']) if rho_hat is not None else Yhat_indep






  dict_preds[val_seasons[0]] = {'Yhat':Yhat, 'Yhat_indep':Yhat_indep, 'Y__SFM':Y__SFMMO, 'rho':rho_hat}


  # ====================================== For In-Sample Evaluation / Sampling Evaluation ====================================== #
  idx_home = complete_data.loc[complete_data['home_pitch'] == 1, 'id_match'].reset_index().set_index('id_match')
  idx_away = complete_data.loc[complete_data['home_pitch'] == 0, 'id_match'].reset_index().set_index('id_match')

  idx_lookup = pd.merge(idx_home,idx_away, left_index=True, right_index=True, suffixes=('_home','_away'))


  dict_fitEval[val_seasons[0]] = {'idata': idata, 'idx_lookup':idx_lookup}



sample: 100%|██████████| 5000/5000 [19:38<00:00,  4.24it/s]
/usr/local/lib/python3.12/dist-packages/jax/_src/interpreters/mlir.py:1334: UserWarning: Some donated buffers were not usable: float64[2,4000,156].
See an explanation at https://docs.jax.dev/en/latest/faq.html#buffer-donation.
  warnings.warn("Some donated buffers were not usable:"


Output()


Scaling Data Cross-Sectionally!



Output()

  0%|          | 0/1399 [00:00<?, ?it/s]/tmp/ipykernel_3297/3402538663.py:778: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  Yhat = pd.concat([Yhat,pd.DataFrame(m_Yhat,index=['0','1','2']).T],axis=0).reset_index(drop=True)
sample: 100%|██████████| 5000/5000 [19:39<00:00,  4.24it/s]
/usr/local/lib/python3.12/dist-packages/jax/_src/interpreters/mlir.py:1334: UserWarning: Some donated buffers were not usable: float64[2,4000,157].
See an explanation at https://docs.jax.dev/en/latest/faq.html#buffer-donation.
  warnings.warn("Some donated buffers were not usable:"


...

  0%|          | 0/1776 [00:00<?, ?it/s]/tmp/ipykernel_3297/3402538663.py:778: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  Yhat = pd.concat([Yhat,pd.DataFrame(m_Yhat,index=['0','1','2']).T],axis=0).reset_index(drop=True)
sample: 100%|██████████| 5000/5000 [20:23<00:00,  4.09it/s]
/usr/local/lib/python3.12/dist-packages/jax/_src/interpreters/mlir.py:1334: UserWarning: Some donated buffers were not usable: float64[2,4000,186].
See an explanation at https://docs.jax.dev/en/latest/faq.html#buffer-donation.
  warnings.warn("Some donated buffers were not usable:"


Output()

Output()


Scaling Data Cross-Sectionally!



  0%|          | 0/1699 [00:00<?, ?it/s]/tmp/ipykernel_3297/3402538663.py:778: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  Yhat = pd.concat([Yhat,pd.DataFrame(m_Yhat,index=['0','1','2']).T],axis=0).reset_index(drop=True)
100%|██████████| 1699/1699 [00:10<00:00, 154.93it/s]


In [ ]:
# =================================== Prior-Predictive Check (M3) =================================== #

# Referee requirement attached to the explicit intercept: verify the priors alone imply a
# sane goals-per-team-game distribution (empirical rate ~1.4) before looking at any fit.
# Uses the LAST fold's model object (exists after the Grand Loop).

with SFMMO__dev:
    prior_pred = pm.sample_prior_predictive(draws=1000, random_seed=seed)

pp_draws = prior_pred.prior_predictive['match_outcome']          # (chain, draw, obs_id)
pp_goals = pp_draws.values.reshape(-1)
per_draw_mean = pp_draws.values.mean(axis=-1).reshape(-1)

print(f"Empirical goals/team-game (train): {complete_data['match_outcome'].mean():.3f}")
print(f"Prior-implied mean               : {pp_goals.mean():.3f}")
print(f"Prior-implied P5-P95 of per-draw means: {np.percentile(per_draw_mean, [5, 95]).round(3)}")

plt.figure(figsize=(7, 4))
plt.hist(pp_goals, bins=np.arange(0, 12) - 0.5, density=True, alpha=0.6, label='prior predictive')
plt.hist(complete_data['match_outcome'], bins=np.arange(0, 12) - 0.5, density=True,
         histtype='step', lw=2, label='observed (train)')
plt.xlabel('goals per team-game'); plt.legend(); plt.title('Prior-predictive check: goals scale')
plt.show()


In [ ]:
# =================================== eta Parity Test (M6a) =================================== #
#
# WHY THIS EXISTS. The deployment pipeline (006_050) does NOT evaluate the PyMC graph: it
# rebuilds lambda in NUMPY from the posterior draws. That hand-assembled formula is exactly
# where the WC2026 `mu` bug lived -- the graph had a global intercept, the reconstruction's
# term list did not. A term present in one path and missing from the other is a SILENT
# MULTIPLICATIVE BIAS (every lambda off by exp(mu)), not an error: nothing crashes, W/D/L
# barely moves, and it survives until someone audits lambda by hand. PyMC cannot catch it --
# both formulas are well-formed, correctly shaped, perfectly sampleable models.
#
# So we assert it numerically: reconstruct eta the way PRODUCTION does, and demand it equals
# the graph's own eta_oos to machine precision. Run this after every spec change.

_post = idata['posterior']
_S = lambda v: _post[v].stack(samples=('chain', 'draw')).values

mu_s     = _S('mu')                       # (S,)
alpha_s  = _S('alpha')                    # (n_teams_train, S)
delta_s  = _S('delta')
bhome_s  = _S('beta_home')
beta_s   = _S('beta')                     # (n_factors, S)

# --- unseen teams live in the PREDICTIONS group (sampled, not fitted) -- concatenate exactly
#     as the graph did (training teams first, then new), or the indices silently misalign.
if n_new > 0:
    _P = oos_preds['predictions']
    _SP = lambda v: _P[v].stack(samples=('chain', 'draw')).values
    alpha_s = np.concatenate([alpha_s, _SP('alpha__new')], axis=0)
    delta_s = np.concatenate([delta_s, _SP('delta__new')], axis=0)
    bhome_s = np.concatenate([bhome_s, _SP('beta_home__new')], axis=0)

_ti = data_oos['name_team'].map(team_to_idx_all).to_numpy()
_oi = data_oos['name_opp'].map(team_to_idx_all).to_numpy()
_Xg = factors_sdz__oos.loc[data_oos.index, factors_g].to_numpy(dtype=float)
_Xh = factors_sdz__oos.loc[data_oos.index, 'home_pitch'].to_numpy(dtype=float)

# --- the PRODUCTION reconstruction (this is the term list 006_050 must also carry)
eta_hat = (mu_s[None, :]
           + alpha_s[_ti]
           - delta_s[_oi]
           + _Xh[:, None] * bhome_s[_ti]
           + _Xg @ beta_s)                                    # (n_obs, S)

eta_graph = oos_preds['predictions']['eta_oos'].stack(samples=('chain', 'draw')).values

_dev = np.abs(eta_hat - eta_graph).max()
print(f"[eta parity] max |reconstruction - graph| = {_dev:.3e}   over {eta_graph.shape[0]:,} rows x {eta_graph.shape[1]:,} draws")
assert _dev < 1e-8, (f"ETA PARITY FAILED ({_dev:.3e}): the NumPy reconstruction disagrees with the "
                     f"model graph. A term is missing from one of them -- this is the mu-bug class. "
                     f"Fix before exporting anything to production.")
print("[eta parity] PASS -- production may rebuild lambda from these draws safely.")

# --- negative control: the test must actually BITE. Drop mu, exactly as the WC pipeline did.
_dev_nomu = np.abs((eta_hat - mu_s[None, :]) - eta_graph).max()
print(f"[eta parity] negative control, mu omitted: max|delta| = {_dev_nomu:.4f} "
      f"-> every lambda deflated by exp(mu) = {np.exp(mu_s.mean()):.3f}x "
      f"({(1 - np.exp(-mu_s.mean()))*100:.1f}% low). That is the bug this cell exists to catch.")


In [10]:
# ================================== Export ================================== #

if 1==1:
  import pickle
  import cloudpickle

  if do__scaleCS:
    scale__type = '_scaleCS' # --- cross-sectional
  else:
    scale__type = '_scaleWS' # --- whole-sample

  _tag = '__HOLDOUT' if RUN_HOLDOUT else ''   # never overwrite the fold pickles (selection evidence)
  pickle_filepath = f'{directory}/10_data/102_Development/Evaluation__SFMMO_Dev{devVersion}_{scale__type}__EW{_tag}.pkl'
  dict_to_save = {
                  'factors': factors,
                  'dict_preds' : dict_preds
                 }

  with open(pickle_filepath , 'wb') as f:
      cloudpickle.dump(dict_to_save, f)


  print(f'Fine. Version: {devVersion}')

Fine. Version: A


<br>

## 2 &emsp; Evaluation -- Fitting


In [ ]:
# =================================== Sampling Diagnostics =================================== #

arviz_stats.diagnose(idata)

In [ ]:
# =================================== Posterior Calibration Diagnostics =================================== #

az.plot_ppc(idata)

azp.plot_loo_pit(idata)

In [ ]:
# =================================== Calibration: Probability of Draws =================================== #

"""
  P(draw) : Observed probability of a draw

"""


# --- For Compatibility:
obs_data = dict_fitEval[val_seasons[0]]['idata']['observed_data'].copy()
idata = dict_fitEval[val_seasons[0]]['idata'].copy()
idx_lookup = dict_fitEval[val_seasons[0]]['idx_lookup'].copy()


# --- Posterior Home Teams:
pp_home = idata['posterior_predictive']['match_outcome'].stack(samples=('chain','draw'))[idx_lookup['index_home'].values,]

# --- Posterior Away Teams:
pp_away = idata['posterior_predictive']['match_outcome'].stack(samples=('chain','draw'))[idx_lookup['index_away'].values,]

# --- Observations Home Teams:
obs_home = complete_data.loc[idx_lookup['index_home'].values,'match_outcome']

# --- Observations Away Teams:
obs_away = complete_data.loc[idx_lookup['index_away'].values,'match_outcome']

# --- Posterior Outcome:
pp_outcome = pp_home.values - pp_away.values

# --- Observations Outcome:
obs_outcome = obs_home.values - obs_away.values



rootogram(obs_home.astype(int).values, pp_home.values, title="Home goals")
rootogram(obs_away.astype(int).values, pp_away.values, title="Away goals")


print(f"Observed P(draw)         : {(obs_outcome == 0).mean():.3f}")
print(f"Posterior predictive mean: {(pp_outcome == 0).mean(axis=0).mean():.3f}")
print(f"94% HDI                  : [{np.quantile((pp_outcome == 0).mean(axis=0), 0.03):.3f}, "
      f"{np.quantile((pp_outcome == 0).mean(axis=0), 0.97):.3f}]")

NameError: name 'dict_fitEval' is not defined

In [ ]:
# =================================== Calibration: Match Outcomes =================================== #


scorelines_obs = list(zip(obs_home, obs_away))
scorelines_pp_per_sample = [
    list(zip(pp_home.values[:, s], pp_away.values[:, s]))
    for s in range(min(pp_home.shape[1], 500))
]

for h, a in [(0, 0), (1, 1), (2, 2), (3, 3), (1, 0), (0, 1), (2, 1), (1, 2)]:
    p_obs = np.mean([s == (h, a) for s in scorelines_obs])
    p_pp  = np.mean([
        np.mean([s == (h, a) for s in sample]) for sample in scorelines_pp_per_sample
    ])
    print(f"{h}-{a}:  obs={p_obs:.4f}  pred={p_pp:.4f}  diff={p_obs - p_pp:+.4f}")


In [ ]:
# =================================== Calibration: Probability of Draws - by League =================================== #


leagues = complete_data.loc[idx_lookup['index_home'].values, "name_league"].to_numpy()
for lg in np.unique(leagues):
    mask = leagues == lg
    p_obs = (obs_outcome[mask] == 0).mean()
    p_pp  = (pp_outcome[mask] == 0).mean(axis=0).mean()
    pp_per_sample = (pp_outcome[mask] == 0).mean(axis=0)
    lo, hi = np.quantile(pp_per_sample, [0.03, 0.97])
    flag = "MISS" if (p_obs < lo or p_obs > hi) else "ok"
    print(f"{lg:>15}  obs={p_obs:.3f}  pred={p_pp:.3f}  HDI=[{lo:.3f}, {hi:.3f}]  {flag}")


<br>

## 3 &emsp; Evaluation -- OOS


In [ ]:
# ================================== Out-of-Sample Evaluation ================================== #
#
# M1 + M7.2: ONE evaluation cell (was seven identical copies compared by eye), reporting
# overall AND **by gameday bucket**. The bucket table is the permanent guard against the
# cold-start blindness: gamedays 1-2 used to be 100% absent from this table because the
# pipeline deleted them, so the regime that failed at WC MD1 could never show up here.
# If a bucket is missing or its n is implausibly small, the harness is lying -- investigate.

BUCKETS = [(1, 2, 'gd 1-2  (cold start)'), (3, 5, 'gd 3-5  (early)'),
           (6, 10, 'gd 6-10'), (11, 20, 'gd 11-20'), (21, 99, 'gd 21+')]

def _gameday_from_id(idm):
    """'<league>_GD07_...' -> 7   (same convention as the Grand Loop)"""
    return int(float(str(idm).split('_')[1][2:]))

def _metrics(P, y):
    y = np.asarray(y, dtype=int)
    return dict(n=len(y),
                logLik=log_loss_categorical(P, y),
                ACC=ordinal_accuracy(P, y),
                RPS=ranked_probability_score(P, y, n_classes=3),
                Brier=multi_class_brier_score(P, y, n_classes=3))

# --- assemble every fold into one long frame (probs + truth + gameday + season)
_P, _Y, _GD, _S = [], [], [], []
for s in dict_preds.keys():
    P = dict_preds[s]['Yhat'].values.astype(float)
    Y = dict_preds[s]['Y__SFM']['match_outcome'].values.astype(int)
    G = dict_preds[s]['Y__SFM']['id_match'].map(_gameday_from_id).values
    _P.append(P); _Y.append(Y); _GD.append(G); _S += [s] * len(Y)
P_all = np.concatenate(_P); Y_all = np.concatenate(_Y)
GD_all = np.concatenate(_GD); S_all = np.array(_S)

# --- per season, then overall
rows = [dict(split=s, **_metrics(P_all[S_all == s], Y_all[S_all == s])) for s in dict_preds.keys()]
rows.append(dict(split='all', **_metrics(P_all, Y_all)))
df_eval = pd.DataFrame(rows).set_index('split')

print(f'\nVersion: {devVersion}   (probability rows sum to {P_all.sum(axis=1).mean():.4f} on average)')
print(df_eval.round(4).to_string())

# --- BY GAMEDAY BUCKET (the M1 guard)
brows = []
for lo, hi, lbl in BUCKETS:
    m = (GD_all >= lo) & (GD_all <= hi)
    if m.sum() == 0:
        brows.append(dict(bucket=lbl, n=0, logLik=np.nan, ACC=np.nan, RPS=np.nan, Brier=np.nan))
        print(f'  !! bucket {lbl!r} is EMPTY -- the pipeline is dropping this regime (see M1).')
        continue
    brows.append(dict(bucket=lbl, **_metrics(P_all[m], Y_all[m])))
df_bucket = pd.DataFrame(brows).set_index('bucket')

# uniform baseline computed on the ACTUAL outcome mix. (Hardcoding 2/9 was a bug: 2/9 is the
# uniform RPS for a DRAW outcome only; home/away outcomes score 5/9, so the weighted mean is
# ~0.47 on this unnormalized scale -- the model was never worse than uniform.)
_unif = np.full_like(P_all, 1.0/3.0)
print(f'\n--- by gameday bucket (uniform baseline: logLik {np.log(3):.4f}, '
      f'RPS {ranked_probability_score(_unif, Y_all, n_classes=3):.4f}) ---')
print(df_bucket.round(4).to_string())

_cold = df_bucket.loc['gd 1-2  (cold start)']
if _cold['n'] > 0 and _cold['logLik'] > np.log(3):
    print(f"\n  !! COLD START WORSE THAN UNIFORM (logLik {_cold['logLik']:.4f} > {np.log(3):.4f}) "
          f"-- this is lesson L1; see season-plan experiment E1.")

# --- E3-B: Dixon-Coles vs independent (same fits; tau applied post-hoc from training-fitted rho)
_k0 = list(dict_preds)[0]
if dict_preds[_k0].get('rho') is not None:
    P_ind = np.concatenate([dict_preds[s]['Yhat_indep'].values.astype(float) for s in dict_preds.keys()])
    print('\n--- E3-B: Dixon-Coles low-score dependence ---')
    print('rho by fold:', {s: round(dict_preds[s]['rho'], 4) for s in dict_preds.keys()})
    for _nm, _PP in [('independent', P_ind), ('Dixon-Coles', P_all)]:
        print(f'  {_nm:12s} logLik {log_loss_categorical(_PP, Y_all):.4f}  '
              f'RPS {ranked_probability_score(_PP, Y_all, n_classes=3):.4f}  '
              f'ACC {ordinal_accuracy(_PP, Y_all):.4f}  '
              f'draw share priced {_PP[:, 1].mean():.4f}')
    print(f'  realized draw share: {(Y_all == 1).mean():.4f}')

df_eval.round(4)
